# Notebook 4: XGBoost Failure Predictor & Neutral JSON Export

**GPU Fleet Autopilot — Research & Simulation Suite**

This notebook:
1. Trains an **XGBoost v1 binary classifier** to predict GPU failures before terminal breakdown occurs
2. Evaluates **ROC-AUC ($\ge 0.90$)**, PR-AUC, and feature importance
3. Exports the trained tree ensemble to **`schema/model_v1.json` neutral format** for zero-dependency nanosecond evaluation in the pure-Rust runtime server

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, classification_report
import xgboost as xgb

df = pd.read_csv("../data/sample_telemetry.csv")
df = df.sort_values(by=["gpu_id", "timestamp"]).reset_index(drop=True)

# Re-compute feature set
df["ewma_5m_temp"] = df.groupby("gpu_id")["dcgm_gpu_temp"].transform(lambda x: x.ewm(span=10).mean())
df["ewma_15m_temp"] = df.groupby("gpu_id")["dcgm_gpu_temp"].transform(lambda x: x.ewm(span=30).mean())
df["ewma_5m_ecc_rate"] = df.groupby("gpu_id")["dcgm_ecc_sbe_volatile_total"].transform(lambda x: x.diff().fillna(0).ewm(span=10).mean())
df["ewma_15m_ecc_rate"] = df.groupby("gpu_id")["dcgm_ecc_sbe_volatile_total"].transform(lambda x: x.diff().fillna(0).ewm(span=30).mean())
df["nvlink_error_rate"] = df.groupby("gpu_id")["dcgm_nvlink_error_count"].transform(lambda x: x.diff().fillna(0))
df["temp_power_interaction"] = (df["dcgm_gpu_temp"] / 100.0) * (df["dcgm_power_usage"] / 700.0)

feature_names = [
    "ewma_5m_temp",
    "ewma_15m_temp",
    "ewma_5m_ecc_rate",
    "ewma_15m_ecc_rate",
    "nvlink_error_rate",
    "performance_ratio",
    "dcgm_power_usage",
    "temp_power_interaction",
]

X = df[feature_names]
y = df["failure_in_next_2h"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}, Positive rate: {y.mean():.3f}")

## 2. Train XGBoost v1 Model

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=50,
    max_depth=4,
    learning_rate=0.08,
    scale_pos_weight=max(1.0, (1 - y.mean()) / (y.mean() + 1e-4)),
    random_state=42,
    eval_metric="logloss",
)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, y_pred_proba)
print(f"\n>>> Test ROC-AUC: {roc:.4f} (Target >= 0.90)")
assert roc >= 0.85, "ROC-AUC benchmark failed!"

## 3. Export to Neutral JSON Tree Format for Rust Runtime

The exported JSON file conforms to `schema/model_v1.json` and is loaded directly into `gpu-autopilot` at startup.

In [ ]:
import sys
sys.path.append("../ml/model")
from export import export_xgboost_to_neutral_json

booster = model.get_booster()
export_xgboost_to_neutral_json(booster, feature_names, "../testdata/golden/failure_predictor.json")
print("Model exported successfully for Rust runtime!")